# Multibanding Mismatch Validation

This notebook confirms that the `MultibandedFrequencyDomain` decimation preserves
waveform fidelity relative to the full-resolution base domain.

## Method

1. Generate A/E/T TDI waveforms on the base `UniformFrequencyDomain`.
2. Decimate to `MultibandedFrequencyDomain` using `decimate()`.
3. Reconstruct a uniform-resolution signal by piecewise-constant interpolation
   (i.e., each multiband bin is broadcast back to the base-domain bins it covers).
4. Compute overlap/mismatch between the original and the reconstructed waveform,
   weighted by the LISA PSD.
5. Sweep over sources drawn from the MBHB prior to obtain a mismatch distribution.

**Run this notebook in the cluster environment** after bootstrapping with
`misc_scripts/lisa_cluster_bootstrap.sh` or equivalent SLURM setup.

In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'misc_scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from misc_scripts.compare_lisa_bbhx_lisabeta import (
    environment_report,
    _require_waveform_stack,
    DEFAULT_LISA_SETTINGS,
    PYCONSTANTS_YRSID_SI,
)

print(environment_report())

In [ ]:
_require_waveform_stack()

from dingo.gw.waveform_generator.waveform_generator import BBHxWaveformGenerator
from dingo.gw.domains import build_domain
from dingo.gw.domains.multibanded_frequency_domain import MultibandedFrequencyDomain
from dingo.gw.domains.uniform_frequency_domain import UniformFrequencyDomain

print('Imports OK')

## Domain setup

In [ ]:
# Base (full-resolution) uniform domain
BASE_DOMAIN_SETTINGS = dict(
    type='UniformFrequencyDomain',
    f_min=1e-4,
    f_max=1e-1,
    delta_f=5e-6,
)
base_domain = build_domain(BASE_DOMAIN_SETTINGS)
freqs_base  = np.array(base_domain.sample_frequencies)
df_base     = base_domain.delta_f
print(f'Base domain: {len(freqs_base)} bins, '
      f'{freqs_base[0]:.2e}–{freqs_base[-1]:.2e} Hz,  Δf = {df_base:.1e} Hz')

# Multibanded domain — same nodes as the training config
#   nodes = band boundaries,  delta_f doubles each band
MBD_NODES        = [1.0e-4, 1.0e-3, 5.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT = 5.0e-6   # Hz  (same as base_domain.delta_f — band 0 = no decimation)

mbd = MultibandedFrequencyDomain(
    nodes=MBD_NODES,
    delta_f_initial=MBD_DELTA_F_INIT,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd = np.array(mbd.sample_frequencies)
print(f'Multibanded domain: {len(freqs_mbd)} bins across {mbd.num_bands} bands')
print(f'  Band Δf values: {[f"{x:.1e}" for x in mbd._delta_f_bands]}')
print(f'  Bins per band:  {list(mbd._num_bins_bands)}')

## Waveform generator

In [ ]:
# Generate on the full MultibandedFrequencyDomain (BBHx uses base_domain internally)
bbhx_gen = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=dict(
        type='MultibandedFrequencyDomain',
        nodes=MBD_NODES,
        delta_f_initial=MBD_DELTA_F_INIT,
        base_domain=BASE_DOMAIN_SETTINGS,
    ),
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

# Separate generator on the base uniform domain for reference
bbhx_gen_base = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=BASE_DOMAIN_SETTINGS,
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

print('Generators initialised')

## LISA PSD and overlap helpers

In [ ]:
C_SI = 3e8

def lisa_psd_A(f, L=2.5e9):
    f   = np.asarray(f, dtype=float)
    S_oms = (1.5e-11)**2 * (1 + (2e-3 / np.maximum(f, 1e-10))**4)
    S_acc = (3e-15)**2 * (1 + (4e-4 / np.maximum(f, 1e-10))**2) * \
            (1 + (f / 8e-3)**4) / (2 * math.pi * np.maximum(f, 1e-10))**4
    S_link = (S_oms + 2 * S_acc) / L**2
    x = 2 * math.pi * f * L / C_SI
    return 8 * np.sin(x)**2 * (2 * (1 + np.cos(x)**2) * S_link)


def inner_product(a, b, psd, df):
    return 4 * df * np.real(np.sum(np.conj(a) * b / psd))


def mismatch(h1, h2, psd, df):
    """1 − overlap, both h1 and h2 on the SAME uniform frequency grid."""
    ov = inner_product(h1, h2, psd, df)
    ov /= math.sqrt(inner_product(h1, h1, psd, df) * inner_product(h2, h2, psd, df))
    return 1.0 - ov


psd_base = lisa_psd_A(freqs_base)
psd_base = np.where(freqs_base > 0, psd_base, np.inf)
print('PSD computed')

## Reconstruct multibanded waveform on the base grid

Each multibanded bin is broadcast (piecewise-constant) back to the base-domain bins
it covers. This mimics how the SVD/embedding network "sees" the compressed data.

In [ ]:
def reconstruct_from_multiband(h_mbd, mbd, base_domain):
    """Piecewise-constant interpolation of a multibanded waveform onto the base grid.
    
    Parameters
    ----------
    h_mbd : ndarray, shape (n_mbd_bins,)
        Complex strain in multibanded domain.
    mbd   : MultibandedFrequencyDomain
    base_domain : UniformFrequencyDomain
    
    Returns
    -------
    h_rec : ndarray, shape (n_base_bins,)
    """
    freqs_base = np.array(base_domain.sample_frequencies)
    freqs_mbd  = np.array(mbd.sample_frequencies)
    # For each base bin, find the MBD bin whose centre frequency is <= base freq.
    # searchsorted(..., 'right') - 1 gives the last MBD bin with freq <= f_base.
    indices = np.searchsorted(freqs_mbd, freqs_base, side='right') - 1
    indices = np.clip(indices, 0, len(freqs_mbd) - 1)
    return h_mbd[indices]


# Quick sanity check on a single waveform
_params_test = dict(
    Mchirp=7e5, q=0.9, chi1=0.3, chi2=0.3,
    inc=0.5, phi=0.0, lam=0.5, beta=0.3, psi=1.2,
    dist=5000.0, geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
)
wf_mbd  = bbhx_gen.generate_amp_phase(_params_test)
wf_base = bbhx_gen_base.generate_amp_phase(_params_test)

# waveform shape is (1, 3, n_freqs): batch=1, channels=3, freqs
h_mbd_chan1  = wf_mbd['waveform'][0][0]   # TDI-A on MBD,  shape (n_mbd,)
h_base_chan1 = wf_base['waveform'][0][0]  # TDI-A on base, shape (n_base,)
h_rec_chan1  = reconstruct_from_multiband(h_mbd_chan1, mbd, base_domain)

mm_test = mismatch(h_base_chan1, h_rec_chan1, psd_base, df_base)
print(f'Single-source mismatch (TDI-A): {mm_test:.4e}')
print(f'  waveform shape: {wf_mbd["waveform"].shape}')

## Visual comparison: amplitude envelope

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax = axes[0]
ax.loglog(freqs_base * 1e3, np.abs(h_base_chan1), lw=1.0, label='Base domain (uniform)', color='k')
ax.loglog(freqs_base * 1e3, np.abs(h_rec_chan1),  lw=0.8, label='Reconstructed from multiband',
          color='steelblue', alpha=0.8, ls='--')
ax.scatter(freqs_mbd * 1e3, np.abs(h_mbd_chan1), s=3, color='red',
           zorder=5, label='Multiband bins')
ax.set_ylabel('|h(f)|  [strain/Hz]')
ax.set_title('TDI-A strain amplitude — base vs. multiband reconstruction')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)

ax = axes[1]
ratio = np.abs(h_rec_chan1) / np.maximum(np.abs(h_base_chan1), 1e-300)
ax.semilogx(freqs_base * 1e3, ratio, lw=0.7, color='steelblue')
ax.axhline(1.0, color='k', ls='--', lw=0.8)
# Mark band boundaries
for node in MBD_NODES[1:-1]:
    ax.axvline(node * 1e3, color='orange', ls=':', lw=1.0, alpha=0.7)
ax.set_ylim(0.5, 1.5)
ax.set_xlabel('Frequency [mHz]')
ax.set_ylabel('|h_rec| / |h_base|')
ax.set_title('Amplitude ratio (orange dashed = band boundaries)')
ax.grid(True, which='both', ls=':', alpha=0.5)

fig.tight_layout()
plt.savefig('multibanding_amplitude_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_amplitude_comparison.png')

## Mismatch distribution over prior

In [ ]:
rng = np.random.default_rng(0)
N_SOURCES = 20

# Sample from the MBHB prior
M_totals = np.exp(rng.uniform(np.log(1e5), np.log(1e6), N_SOURCES))

mismatches_chan1 = []
mismatches_chan2 = []

for M_tot in M_totals:
    q  = rng.uniform(0.1, 1.0)
    eta = q / (1 + q)**2
    Mchirp = M_tot * eta**0.6
    params = dict(
        Mchirp=Mchirp, q=q,
        chi1=rng.uniform(-0.9, 0.9),
        chi2=rng.uniform(-0.9, 0.9),
        inc=np.arccos(rng.uniform(-1, 1)),
        phi=rng.uniform(0, 2 * math.pi),
        lam=rng.uniform(0, 2 * math.pi),
        beta=np.arcsin(rng.uniform(-1, 1)),
        psi=rng.uniform(0, math.pi),
        dist=rng.uniform(500, 20000),
        geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
    )
    try:
        wf_m = bbhx_gen.generate_amp_phase(params)
        wf_b = bbhx_gen_base.generate_amp_phase(params)
        # waveform shape: (1, 3, n_freqs) — index [0][ch_idx] to get single channel
        for ch_idx, mm_list in [(0, mismatches_chan1), (1, mismatches_chan2)]:
            h_m   = wf_m['waveform'][0][ch_idx]
            h_b   = wf_b['waveform'][0][ch_idx]
            h_rec = reconstruct_from_multiband(h_m, mbd, base_domain)
            mm_list.append(mismatch(h_b, h_rec, psd_base, df_base))
    except Exception as e:
        print(f'  M_tot={M_tot:.2e}: {e}')

mismatches_chan1 = np.array(mismatches_chan1)
mismatches_chan2 = np.array(mismatches_chan2)

print(f'TDI-A mismatch: median={np.median(mismatches_chan1):.3e},  '
      f'90th pct={np.percentile(mismatches_chan1, 90):.3e},  '
      f'max={mismatches_chan1.max():.3e}')
print(f'TDI-E mismatch: median={np.median(mismatches_chan2):.3e},  '
      f'90th pct={np.percentile(mismatches_chan2, 90):.3e},  '
      f'max={mismatches_chan2.max():.3e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bins = np.logspace(np.log10(1e-6), np.log10(0.1), 30)
ax.hist(mismatches_chan1, bins=bins, alpha=0.6, label='TDI-A', color='steelblue')
ax.hist(mismatches_chan2, bins=bins, alpha=0.6, label='TDI-E', color='darkorange')
ax.axvline(1e-3, color='red', ls='--', lw=1.2, label='0.1% threshold')
ax.set_xscale('log')
ax.set_xlabel('Mismatch  (1 − overlap)')
ax.set_ylabel('Count')
ax.set_title(f'Multibanding mismatch distribution ({N_SOURCES} sources)')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
fig.tight_layout()
plt.savefig('multibanding_mismatch_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_mismatch_distribution.png')

## Sensitivity to node placement

Try a coarser multibanding scheme and compare mismatch, to understand the trade-off
between compression ratio and fidelity.

In [ ]:
# Coarser alternative: fewer bands, larger delta_f_initial
MBD_NODES_COARSE        = [1.0e-4, 2.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT_COARSE = 2.0e-5  # 20 μHz (4× coarser initial band)

mbd_coarse = MultibandedFrequencyDomain(
    nodes=MBD_NODES_COARSE,
    delta_f_initial=MBD_DELTA_F_INIT_COARSE,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd_coarse = np.array(mbd_coarse.sample_frequencies)
print(f'Coarse MBD: {len(freqs_mbd_coarse)} bins across {mbd_coarse.num_bands} bands')
print(f'  Bins per band: {list(mbd_coarse._num_bins_bands)}')

# Quick mismatch check on same test source — decimate from base domain waveform
# waveform shape: (1, 3, n_freqs); [0][0] selects TDI-A
wf_base_test = bbhx_gen_base.generate_amp_phase(_params_test)
h_mbd_c  = mbd_coarse.decimate(wf_base_test['waveform'][0][0])
h_rec_c  = reconstruct_from_multiband(h_mbd_c, mbd_coarse, base_domain)
mm_coarse = mismatch(h_base_chan1, h_rec_c, psd_base, df_base)
print(f'Coarse scheme single-source mismatch (TDI-A): {mm_coarse:.4e}')
print(f'Fine   scheme single-source mismatch (TDI-A): {mm_test:.4e}')

## Summary

- If median mismatch is < 0.1% (1e-3) with the chosen multiband nodes, the
  configuration is acceptable for training.
- If mismatch is too high (> 0.1%), either reduce `delta_f_initial` (finer first band)
  or adjust node positions to avoid sharp band transitions near the signal peak.
- The coarser scheme shows how aggressively we can compress before fidelity degrades.

Adjust `MBD_NODES` and `MBD_DELTA_F_INIT` in `waveform_dataset_settings.yaml` based
on these results.